# Template: carga de datos de siniestros viales

Este notebook es una plantilla para cargar los archivos xlsx de `data/raw`.

**Antes de empezar**

1. Ejecuta `git pull` y `dvc pull` desde la raíz del proyecto para tener los datos en `data/raw`.
2. Guarda este notebook en la carpeta `notebooks/` y abre Jupyter con el kernel del proyecto.
3. No edites los archivos de `data/raw`: todo cambio se hace aquí y se guarda en `data/processed`.

**Cómo usarlo**

- **Opción A:** cargar **un solo archivo** (y la hoja que elijas). Úsala cuando los archivos no comparten el mismo esquema.
- **Opción B:** cargar **todos los archivos** y comparar sus columnas antes de unirlos.

## 0. Configuración y rutas

In [1]:
from pathlib import Path
import pandas as pd
import siniestros_loader as sl
from pathlib import Path


pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

# Raíz del proyecto: este notebook vive en notebooks/, así que la raíz es la carpeta superior.
# Si guardas el notebook en la raíz, cambia la línea por: ROOT = Path.cwd()
ROOT = Path.cwd().parent
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"

print("Carpeta de datos:", RAW)
print("¿Existe?", RAW.exists())

Carpeta de datos: d:\Documents\Courses\Data Analytics\Python\8. Python DataCamp\18.Web Scraping\Repository\ds_project_dvc_MIAD\data\raw
¿Existe? True


## 1. Ver los archivos y sus hojas

Revisa qué archivos hay y cuántas hojas tiene cada uno. Si la carpeta aparece vacía, ejecuta `dvc pull` desde la raíz del proyecto.

In [9]:
archivos = sorted(RAW.glob("*.xlsx"))
print(f"{len(archivos)} archivos encontrados\n")

for i, f in enumerate(archivos):
    hojas = pd.ExcelFile(f).sheet_names
    print(f"[{i}] {f.name}  ->  hojas: {hojas}")

7 archivos encontrados

[0] Base_Anuario_de_Siniestralidad_Vial_2019.xlsx  ->  hojas: ['SINIESTROS', 'VEHICULOS', 'ACTOR_VIAL']
[1] Base_Anuario_de_Siniestralidad_Vial_2020.xlsx  ->  hojas: ['SINIESTROS', 'VEHICULOS', 'ACTOR_VIAL', 'DICCIONARIO']
[2] Base_Anuario_de_Siniestralidad_Vial_2021.xlsx  ->  hojas: ['SINIESTROS', 'VEHICULOS', 'ACTOR_VIAL', 'DICCIONARIO']
[3] Base_Anuario_de_Siniestralidad_Vial_2022.xlsx  ->  hojas: ['SINIESTROS', 'VEHICULOS', 'ACTOR_VIAL', 'DICCIONARIO']
[4] Base_Anuario_de_Siniestralidad_Vial_2023.xlsx  ->  hojas: ['Siniestros', 'Vehiculos', 'Actor_vial', 'Diccionario']
[5] Base_Anuario_de_Siniestralidad_Vial_2024.xlsx  ->  hojas: ['Siniestros', 'Vehiculos', 'Actor_vial', 'Diccionario']
[6] Base_Anuario_de_Siniestralidad_Vial_2025.xlsx  ->  hojas: ['Siniestros', 'Vehículos', 'Actor vial', 'Diccionario']


## 2. Opción A: cargar un solo archivo

Elige el archivo con su **índice** (el número entre corchetes de la celda anterior) o con su **nombre**, y la **hoja** (por posición o por nombre).

Si la tabla no empieza en la primera fila de la hoja, usa `HEADER` para indicar en qué fila están los nombres de las columnas (empieza en 0).

In [22]:
# --- Parámetros: cambia estos valores ---
INDICE_ARCHIVO = 0          # posición en la lista de archivos
NOMBRE_ARCHIVO = 'Base_Anuario_de_Siniestralidad_Vial_2021.xlsx'       # o el nombre exacto, por ejemplo "siniestros_2019.xlsx" (tiene prioridad si no es None)
HOJA = 'DICCIONARIO'                    # número de hoja (0 = primera) o su nombre
HEADER = 1                 # fila con los nombres de columnas
# ----------------------------------------

ruta = RAW / NOMBRE_ARCHIVO if NOMBRE_ARCHIVO else archivos[INDICE_ARCHIVO]

df = pd.read_excel(ruta, sheet_name=HOJA, header=HEADER)
df["source_file"] = ruta.name

print("Archivo:", ruta.name)
print("Filas y columnas:", df.shape)
df.head()

Archivo: Base_Anuario_de_Siniestralidad_Vial_2021.xlsx
Filas y columnas: (102, 5)


,SINIESTROS,GRAVEDAD,1,Con Muertos,source_file
0,SINIESTROS,GRAVEDAD,2,Con Heridos,Base_Anuario_de_Siniestralidad_Vial_2021.xlsx
1,SINIESTROS,GRAVEDAD,3,Solo Daños,Base_Anuario_de_Siniestralidad_Vial_2021.xlsx
2,SINIESTROS,CLASE,7,Autolesion,Base_Anuario_de_Siniestralidad_Vial_2021.xlsx
3,SINIESTROS,CLASE,1,Choque,Base_Anuario_de_Siniestralidad_Vial_2021.xlsx
4,SINIESTROS,CLASE,2,Atropello,Base_Anuario_de_Siniestralidad_Vial_2021.xlsx


In [24]:
df_diccionario = df

OUT = Path(r"C:\Users\oscar\OneDrive - Universidad de los Andes\Posgrado\04. Despliegue de soluciones analíticas\Datos limpios\diccionario.parquet")
OUT.parent.mkdir(parents=True, exist_ok=True)   # creates the folder if missing

df.to_parquet(OUT, index=False)

In [5]:
# Estructura del archivo cargado
print(df.columns.tolist())
df.dtypes

['Accidente', 'Formulario', 'FECHA', 'HORA', 'MINUTO', 'GravedadNombre', 'ClaseCodigo', 'ChoqueNombre', 'ObjetoFijoNombre', 'Direccion', 'Localidad', 'CON_BICICLETA', 'CON_CARGA', 'CON_EMBRIAGUEZ', 'CON_HUECOS', 'CON_MENORES', 'CON_MOTO', 'CON_PEATON', 'CON_PERSONA_MAYOR', 'CON_RUTAS', 'CON_TPI', 'CON_VELOCIDAD', 'CON_ZONAL', 'CON_TRONCAL', 'CON_ALIMENTADOR', 'CON_PROVISIONAL', 'source_file']


Accidente                     int64
Formulario                      str
FECHA                datetime64[us]
HORA                          int64
MINUTO                        int64
GravedadNombre                  str
ClaseCodigo                   int64
ChoqueNombre                    str
ObjetoFijoNombre                str
Direccion                       str
Localidad                       str
CON_BICICLETA                   str
CON_CARGA                       str
CON_EMBRIAGUEZ                  str
CON_HUECOS                      str
CON_MENORES                     str
CON_MOTO                        str
CON_PEATON                      str
CON_PERSONA_MAYOR               str
CON_RUTAS                       str
CON_TPI                         str
CON_VELOCIDAD                   str
CON_ZONAL                   float64
CON_TRONCAL                 float64
CON_ALIMENTADOR             float64
CON_PROVISIONAL             float64
source_file                     str
dtype: object

## 3. Opción B: cargar todos los archivos y comparar esquemas

Esta opción lee **una hoja de cada archivo** y guarda cada resultado por separado, **sin unirlos todavía**. Si un archivo falla, lo reporta y sigue con el resto.

In [6]:
def cargar_todos(hoja=0, header=0):
    """Devuelve un diccionario {nombre_archivo: DataFrame}."""
    resultado = {}
    for f in sorted(RAW.glob("*.xlsx")):
        try:
            d = pd.read_excel(f, sheet_name=hoja, header=header)
            d["source_file"] = f.name
            resultado[f.name] = d
        except Exception as e:
            print(f"No se pudo leer {f.name}: {e}")
    return resultado


dfs = cargar_todos(hoja=0)   # cambia 'hoja' si los datos no están en la primera hoja

for nombre, d in dfs.items():
    print(f"{nombre}: {d.shape[0]} filas, {d.shape[1]} columnas")

Base_Anuario_de_Siniestralidad_Vial_2019.xlsx: 34990 filas, 27 columnas
Base_Anuario_de_Siniestralidad_Vial_2020.xlsx: 22712 filas, 33 columnas
Base_Anuario_de_Siniestralidad_Vial_2021.xlsx: 28841 filas, 34 columnas
Base_Anuario_de_Siniestralidad_Vial_2022.xlsx: 25447 filas, 45 columnas
Base_Anuario_de_Siniestralidad_Vial_2023.xlsx: 14106 filas, 39 columnas
Base_Anuario_de_Siniestralidad_Vial_2024.xlsx: 278614 filas, 41 columnas
Base_Anuario_de_Siniestralidad_Vial_2025.xlsx: 11573 filas, 26 columnas


### Comparar los esquemas

Muestra qué columnas comparte cada archivo con los demás. Las diferencias son las que debes resolver (renombrar columnas, descartar hojas, etc.) antes de unir los datos.

In [7]:
columnas = {nombre: set(d.columns) - {"source_file"} for nombre, d in dfs.items()}

comunes = set.intersection(*columnas.values()) if columnas else set()
print(f"Columnas comunes a todos los archivos ({len(comunes)}):")
print(sorted(comunes))

print("\nColumnas adicionales por archivo (no comunes a todos):")
for nombre, cols in columnas.items():
    extra = sorted(cols - comunes)
    if extra:
        print(f"- {nombre}: {extra}")


Columnas comunes a todos los archivos (0):
[]

Columnas adicionales por archivo (no comunes a todos):
- Base_Anuario_de_Siniestralidad_Vial_2019.xlsx: ['Accidente', 'CON_ALIMENTADOR', 'CON_BICICLETA', 'CON_CARGA', 'CON_EMBRIAGUEZ', 'CON_HUECOS', 'CON_MENORES', 'CON_MOTO', 'CON_PEATON', 'CON_PERSONA_MAYOR', 'CON_PROVISIONAL', 'CON_RUTAS', 'CON_TPI', 'CON_TRONCAL', 'CON_VELOCIDAD', 'CON_ZONAL', 'ChoqueNombre', 'ClaseCodigo', 'Direccion', 'FECHA', 'Formulario', 'GravedadNombre', 'HORA', 'Localidad', 'MINUTO', 'ObjetoFijoNombre']
- Base_Anuario_de_Siniestralidad_Vial_2020.xlsx: ['CHOQUE', 'CLASE', 'CODIGO_ACCIDENTE', 'CODIGO_LOCALIDAD', 'CON_ALIMENTADOR', 'CON_BICICLETA', 'CON_CARGA', 'CON_EMBRIAGUEZ', 'CON_HUECOS', 'CON_MENORES', 'CON_MOTO', 'CON_PADRON_DUAL', 'CON_PEATON', 'CON_PERSONA_MAYOR', 'CON_PROVISIONAL', 'CON_RUTAS', 'CON_SITP', 'CON_TPI', 'CON_TPP', 'CON_TRANSMILENIO', 'CON_TRONCAL', 'CON_VELOCIDAD', 'CON_ZONAL', 'DIRECCION', 'DISENO_LUGAR', 'FECHA', 'FORMULARIO', 'GravedadNombr

## 4. Unir los archivos (opcional)

Haz esto **solo después** de revisar los esquemas. Dos alternativas:

- **Solo columnas comunes:** seguro, pero pierdes las columnas que no comparten.
- **Todas las columnas:** `pd.concat` crea las columnas que falten y las deja vacías (`NaN`).

Si los nombres de una misma variable cambian entre años (por ejemplo `FECHA` y `Fecha`), renómbralos antes con un diccionario.

In [ ]:
# Renombrar columnas que cambian entre años (ejemplo; ajusta según lo que encuentres)
RENOMBRAR = {
    # "Fecha": "FECHA",
    # "Localidad ": "LOCALIDAD",
}

frames = [d.rename(columns=RENOMBRAR) for d in dfs.values()]

SOLO_COMUNES = True   # False para conservar todas las columnas

if SOLO_COMUNES:
    comunes = set.intersection(*[set(f.columns) for f in frames])
    frames = [f[[c for c in f.columns if c in comunes]] for f in frames]

data = pd.concat(frames, ignore_index=True)
print("Filas y columnas del conjunto unido:", data.shape)
data.head()

## 5. Guardar el resultado

Guarda el resultado en `data/processed`, **nunca** en `data/raw`. Esa carpeta la versiona DVC (ver el README del repositorio).

In [ ]:
PROCESSED.mkdir(parents=True, exist_ok=True)

# Descomenta la línea que necesites:
# data.to_csv(PROCESSED / "siniestros.csv", index=False)
# df.to_csv(PROCESSED / "siniestros_un_archivo.csv", index=False)

# Consolidacion de los documentos de excel

## Consolidacion de siniestros

In [10]:


OUT_PARQUET = Path(r"C:\Users\oscar\OneDrive - Universidad de los Andes\Posgrado\04. Despliegue de soluciones analíticas\Datos limpios\siniestros.parquet")

SHEET = "siniestros"

df = sl.build_siniestros(
    RAW,
    sheet=SHEET,
    interim_dir=PROCESSED / "interim",
    out_path=OUT_PARQUET,
)
sl.quick_checks(df)

  Base_Anuario_de_Siniestralidad_Vial_2019.xlsx: sheet 'SINIESTROS' -> 34,990 rows x 26 cols
  Base_Anuario_de_Siniestralidad_Vial_2020.xlsx: sheet 'SINIESTROS' -> 22,712 rows x 32 cols
  Base_Anuario_de_Siniestralidad_Vial_2021.xlsx: sheet 'SINIESTROS' -> 28,841 rows x 33 cols
  Base_Anuario_de_Siniestralidad_Vial_2022.xlsx: sheet 'SINIESTROS' -> 25,447 rows x 44 cols
  Base_Anuario_de_Siniestralidad_Vial_2023.xlsx: sheet 'Siniestros' -> 14,106 rows x 38 cols
  Base_Anuario_de_Siniestralidad_Vial_2024.xlsx: sheet 'Siniestros' -> 278,614 rows x 40 cols
  Base_Anuario_de_Siniestralidad_Vial_2025.xlsx: sheet 'Siniestros' -> 11,573 rows x 25 cols
[clase] learned code->name: {1: 'CHOQUE', 2: 'ATROPELLO', 4: 'CAIDA DE OCUPANTE', 6: 'OTRO', 3: 'VOLCAMIENTO', 5: 'INCENDIO'} | rows still missing their pair: 14
[choque] learned code->name: {1: 'VEHICULO', 4: 'OBJETO FIJO', 3: 'SEMOVIENTE', 2: 'TREN'} | rows still missing their pair: 55
[objeto_fijo] learned code->name: {} | rows still missing t

In [11]:


DATA_DIR = Path(r"C:\Users\oscar\OneDrive - Universidad de los Andes\Posgrado\04. Despliegue de soluciones analíticas\Datos limpios")

archivos = sorted(DATA_DIR.glob("*.parquet"))
print(f"{len(archivos)} archivos parquet encontrados\n")

tablas = {}
for f in archivos:
    tablas[f.stem] = pd.read_parquet(f)
    print(f"{f.name}: {tablas[f.stem].shape[0]:,} filas x {tablas[f.stem].shape[1]} columnas")

# Access each table by its file name:
df_siniestros = tablas["siniestros"]

1 archivos parquet encontrados

siniestros.parquet: 416,283 filas x 51 columnas


In [12]:
df_siniestros.head()

,source_year,source_file,codigo_siniestro,formulario,fecha,hora,minuto,fecha_hora,anio,mes,dia,dia_semana,direccion,localidad_codigo,localidad,diseno_lugar_codigo,longitud,latitud,clase_codigo,clase_nombre,choque_codigo,choque_nombre,objeto_fijo_codigo,objeto_fijo_nombre,gravedad_tradicional_codigo,gravedad_tradicional_nombre,gravedad_30d_codigo,gravedad_30d_nombre,con_bicicleta,con_carga,con_embriaguez,con_huecos,con_menores,con_moto,con_peaton,con_persona_mayor,con_rutas,con_tpi,con_tpp,con_velocidad,con_sitp,con_transmilenio,con_troncal,con_alimentador,con_zonal,con_provisional,con_articulado,con_biarticulado,con_padron_dual,con_servicio_especial,con_taxi
0,2019,Base_Anuario_de_Siniestralidad_Vial_2019.xlsx,10473344,A000901730,2019-01-21,22,10,2019-01-21 22:10:00,2019,1,21,LUNES,AK 10-CL 11 2,<NA>,SANTA FE,<NA>,NaN,NaN,2,ATROPELLO,<NA>,<NA>,<NA>,<NA>,2,CON HERIDOS,<NA>,<NA>,False,False,False,False,False,False,True,True,False,False,<NA>,False,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,2019,Base_Anuario_de_Siniestralidad_Vial_2019.xlsx,10472331,A000902035,2019-01-08,19,20,2019-01-08 19:20:00,2019,1,8,MARTES,KR 4-CL 29 2,<NA>,SANTA FE,<NA>,NaN,NaN,1,CHOQUE,1,VEHICULO,<NA>,<NA>,3,SOLO DANOS,<NA>,<NA>,False,False,False,False,False,True,False,True,False,False,<NA>,False,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,2019,Base_Anuario_de_Siniestralidad_Vial_2019.xlsx,10484682,A000902364,2019-01-01,9,45,2019-01-01 09:45:00,2019,1,1,MARTES,CL 78-KR 80J S 2,<NA>,BOSA,<NA>,NaN,NaN,1,CHOQUE,1,VEHICULO,<NA>,<NA>,2,CON HERIDOS,<NA>,<NA>,True,False,False,False,False,False,False,False,False,False,<NA>,False,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,2019,Base_Anuario_de_Siniestralidad_Vial_2019.xlsx,10484322,A000902583,2019-01-01,11,30,2019-01-01 11:30:00,2019,1,1,MARTES,CL 100-KR 67A 2,<NA>,BARRIOS UNIDOS,<NA>,NaN,NaN,1,CHOQUE,1,VEHICULO,<NA>,<NA>,3,SOLO DANOS,<NA>,<NA>,False,False,False,False,False,True,False,False,False,False,<NA>,False,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,2019,Base_Anuario_de_Siniestralidad_Vial_2019.xlsx,10473366,A000902649,2019-01-18,16,50,2019-01-18 16:50:00,2019,1,18,VIERNES,AV AVENIDA DEL SUR-CL 59 02,<NA>,BOSA,<NA>,NaN,NaN,1,CHOQUE,1,VEHICULO,<NA>,<NA>,3,SOLO DANOS,<NA>,<NA>,False,False,False,False,False,False,False,False,False,False,<NA>,False,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


## Consolidacion de vehiculos impactados

In [3]:
OUT_PARQUET = Path(r"C:\Users\oscar\OneDrive - Universidad de los Andes\Posgrado\04. Despliegue de soluciones analíticas\Datos limpios\vehiculos.parquet")

SHEET = "vehiculos"

df = sl.build_vehiculos(
    RAW,
    sheet=SHEET,
    interim_dir=PROCESSED / "interim",
    out_path=OUT_PARQUET,
    force=True,
)
sl.quick_checks(df, table="vehiculos")

  Base_Anuario_de_Siniestralidad_Vial_2019.xlsx: sheet 'VEHICULOS' -> 66,179 rows x 22 cols
  Base_Anuario_de_Siniestralidad_Vial_2020.xlsx: sheet 'VEHICULOS' -> 43,220 rows x 25 cols
  Base_Anuario_de_Siniestralidad_Vial_2021.xlsx: sheet 'VEHICULOS' -> 55,097 rows x 26 cols
  Base_Anuario_de_Siniestralidad_Vial_2022.xlsx: sheet 'VEHICULOS' -> 47,619 rows x 29 cols
  Base_Anuario_de_Siniestralidad_Vial_2023.xlsx: sheet 'Vehiculos' -> 24,403 rows x 30 cols
  Base_Anuario_de_Siniestralidad_Vial_2024.xlsx: sheet 'Vehiculos' -> 523,168 rows x 32 cols
  Base_Anuario_de_Siniestralidad_Vial_2025.xlsx: sheet 'Vehículos' -> 19,974 rows x 18 cols
[clase_vehiculo] code->name known: {}
    codes with NO name yet (code: years) -> look them up in DICCIONARIO: {0: [2020, 2021], 1: [2020, 2021], 2: [2020, 2021], 3: [2020, 2021], 4: [2020, 2021], 5: [2020, 2021], 6: [2020, 2021], 7: [2020, 2021], 8: [2020, 2021], 9: [2020, 2021], 10: [2020, 2021], 11: [2021], 12: [2021], 13: [2020, 2021], 14: [2020, 20

In [4]:
DATA_DIR = Path(r"C:\Users\oscar\OneDrive - Universidad de los Andes\Posgrado\04. Despliegue de soluciones analíticas\Datos limpios")

archivos = sorted(DATA_DIR.glob("*.parquet"))
print(f"{len(archivos)} archivos parquet encontrados\n")

tablas = {}
for f in archivos:
    tablas[f.stem] = pd.read_parquet(f)
    print(f"{f.name}: {tablas[f.stem].shape[0]:,} filas x {tablas[f.stem].shape[1]} columnas")

# Access each table by its file name:
df_vehiculos = tablas["vehiculos"]

2 archivos parquet encontrados

siniestros.parquet: 416,283 filas x 51 columnas
vehiculos.parquet: 779,660 filas x 39 columnas


## Consolidacion de actores viales

In [5]:
OUT_ACTOR = Path(r"C:\Users\oscar\OneDrive - Universidad de los Andes\Posgrado\04. Despliegue de soluciones analíticas\Datos limpios\actor_vial.parquet")

df_actor = sl.build_actor_vial(
    RAW,
    interim_dir=PROCESSED / "interim",
    out_path=OUT_ACTOR,
)
sl.quick_checks(df_actor, table="actor_vial")

  Base_Anuario_de_Siniestralidad_Vial_2019.xlsx: sheet 'ACTOR_VIAL' -> 75,644 rows x 26 cols
  Base_Anuario_de_Siniestralidad_Vial_2020.xlsx: sheet 'ACTOR_VIAL' -> 48,345 rows x 33 cols
  Base_Anuario_de_Siniestralidad_Vial_2021.xlsx: sheet 'ACTOR_VIAL' -> 62,364 rows x 34 cols
  Base_Anuario_de_Siniestralidad_Vial_2022.xlsx: sheet 'ACTOR_VIAL' -> 56,443 rows x 43 cols
  Base_Anuario_de_Siniestralidad_Vial_2023.xlsx: sheet 'Actor_vial' -> 33,542 rows x 42 cols
  Base_Anuario_de_Siniestralidad_Vial_2024.xlsx: sheet 'Actor_vial' -> 608,155 rows x 44 cols
  Base_Anuario_de_Siniestralidad_Vial_2025.xlsx: sheet 'Actor vial' -> 28,029 rows x 31 cols
Saved 912,522 rows x 54 cols -> C:\Users\oscar\OneDrive - Universidad de los Andes\Posgrado\04. Despliegue de soluciones analíticas\Datos limpios\actor_vial.parquet

[actor_vial] rows per source file year:
source_year
2019     75644
2020     48345
2021     62364
2022     56443
2023     33542
2024    608155
2025     28029

Event year (from fecha) 

In [6]:
DATA_DIR = Path(r"C:\Users\oscar\OneDrive - Universidad de los Andes\Posgrado\04. Despliegue de soluciones analíticas\Datos limpios")

archivos = sorted(DATA_DIR.glob("*.parquet"))
print(f"{len(archivos)} archivos parquet encontrados\n")

tablas = {}
for f in archivos:
    tablas[f.stem] = pd.read_parquet(f)
    print(f"{f.name}: {tablas[f.stem].shape[0]:,} filas x {tablas[f.stem].shape[1]} columnas")

# Access each table by its file name:
df_actor_vial = tablas["actor_vial"]

3 archivos parquet encontrados

actor_vial.parquet: 912,522 filas x 54 columnas
siniestros.parquet: 416,283 filas x 51 columnas
vehiculos.parquet: 779,660 filas x 39 columnas


In [7]:
df_actor_vial.head()

,source_year,source_file,codigo_siniestro,codigo_accidentado,codigo_vehiculo,codigo_victima,formulario,fecha,hora,minuto,fecha_hora,anio,mes,dia,dia_semana,localidad,edad,sexo,gravedad_tradicional,gravedad_30d,muerte_posterior,fecha_cambio_gravedad,clinica_atencion,condicion,condicion_actv,condicion_especifica,tipo_sitp,victima_categoria,asociado_categoria,con_cinturon,con_casco,con_bicicleta,con_carga,con_embriaguez,con_huecos,con_menores,con_moto,con_peaton,con_persona_mayor,con_rutas,con_tpi,con_tpp,con_velocidad,con_sitp,con_transmilenio,con_troncal,con_alimentador,con_zonal,con_provisional,con_articulado,con_biarticulado,con_padron_dual,con_servicio_especial,con_taxi
0,2019,Base_Anuario_de_Siniestralidad_Vial_2019.xlsx,10473344,<NA>,1,<NA>,A000901730,2019-01-21,22,10,2019-01-21 22:10:00,2019,1,21,LUNES,<NA>,<NA>,NO APLICA,ILESO,<NA>,<NA>,NaT,<NA>,CONDUCTOR,<NA>,COND VEHICULO SIN INFO,<NA>,<NA>,<NA>,<NA>,<NA>,False,False,False,False,False,False,True,True,False,False,<NA>,False,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,2019,Base_Anuario_de_Siniestralidad_Vial_2019.xlsx,10472331,<NA>,1,<NA>,A000902035,2019-01-08,19,20,2019-01-08 19:20:00,2019,1,8,MARTES,<NA>,23,MASCULINO,ILESO,<NA>,<NA>,NaT,<NA>,MOTOCICLISTA,<NA>,MOTOCICLISTA,<NA>,<NA>,<NA>,<NA>,<NA>,False,False,False,False,False,True,False,True,False,False,<NA>,False,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,2019,Base_Anuario_de_Siniestralidad_Vial_2019.xlsx,10472331,<NA>,2,<NA>,A000902035,2019-01-08,19,20,2019-01-08 19:20:00,2019,1,8,MARTES,<NA>,<NA>,NO APLICA,ILESO,<NA>,<NA>,NaT,<NA>,CONDUCTOR,<NA>,COND VEHICULO SIN INFO,<NA>,<NA>,<NA>,<NA>,<NA>,False,False,False,False,False,True,False,True,False,False,<NA>,False,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,2019,Base_Anuario_de_Siniestralidad_Vial_2019.xlsx,10484682,<NA>,2,<NA>,A000902364,2019-01-01,9,45,2019-01-01 09:45:00,2019,1,1,MARTES,<NA>,26,MASCULINO,HERIDO VALORADO,<NA>,<NA>,NaT,<NA>,CICLISTA,<NA>,CICLISTA,<NA>,<NA>,<NA>,<NA>,<NA>,True,False,False,False,False,False,False,False,False,False,<NA>,False,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,2019,Base_Anuario_de_Siniestralidad_Vial_2019.xlsx,10484682,<NA>,1,<NA>,A000902364,2019-01-01,9,45,2019-01-01 09:45:00,2019,1,1,MARTES,<NA>,28,MASCULINO,ILESO,<NA>,<NA>,NaT,<NA>,CONDUCTOR,<NA>,COND TPP,<NA>,<NA>,<NA>,<NA>,<NA>,True,False,False,False,False,False,False,False,False,False,<NA>,False,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
